In [1]:
# Install necessary libraries
!pip install -q xgboost ipywidgets pandas numpy scikit-learn matplotlib

import numpy as np
import pandas as pd
import datetime
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Define neighboring suburbs and intersections data
regions_data = {
    "Riverside": [
        {"name": "Longcommon Rd & E Burlington St", "lat": 41.8295, "lon": -87.8182},
        {"name": "Riverside Rd & Bloomingbank Rd", "lat": 41.8312, "lon": -87.8225},
        {"name": "First Ave & 31st St Intersection", "lat": 41.8385, "lon": -87.8341}
    ],
    "Brookfield": [
        {"name": "Ogden Ave & Grand Blvd", "lat": 41.8235, "lon": -87.8440},
        {"name": "Custer Ave & 47th St", "lat": 41.8080, "lon": -87.8490},
        {"name": "Brookfield Ave & 31st St", "lat": 41.8350, "lon": -87.8420}
    ],
    "North Riverside": [
        {"name": "Cermak Rd & Harlem Ave", "lat": 41.8520, "lon": -87.8050},
        {"name": "Desplaines Ave & 26th St", "lat": 41.8400, "lon": -87.8150},
        {"name": "Appletree Ln & 9th Ave", "lat": 41.8480, "lon": -87.8180}
    ],
    "La Grange": [
        {"name": "La Grange Rd & Burlington Ave", "lat": 41.8150, "lon": -87.8710},
        {"name": "Ogden Ave & Kensington Ave", "lat": 41.8190, "lon": -87.8630},
        {"name": "Wabash Ave & 47th St", "lat": 41.8090, "lon": -87.8680}
    ]
}

# Generate synthetic dataset covering all regions
np.random.seed(42)
n_samples = 4000
date_start = datetime.date(2025, 1, 1)
date_end = datetime.date(2026, 12, 31)
delta_days = (date_end - date_start).days

all_rows = []
for _ in range(n_samples):
    reg_name = np.random.choice(list(regions_data.keys()))
    loc = np.random.choice(regions_data[reg_name])
    random_days = np.random.randint(0, delta_days)
    record_date = date_start + datetime.timedelta(days=random_days)

    hour = np.random.randint(0, 24)
    day_of_week = record_date.weekday()
    temperature = np.random.normal(15, 10)
    precipitation = np.random.choice([0.0, 0.5, 2.5, 10.0], p=[0.7, 0.2, 0.08, 0.02])
    traffic_flow = np.random.randint(100, 1500)
    is_roadwork = np.random.choice([0, 1], p=[0.85, 0.15])

    risk_score = (
        (1.5 if (7 <= hour <= 9 or 17 <= hour <= 19) else 0.5) * 0.3 +
        (1.2 if day_of_week < 5 else 0.8) * 0.1 +
        (1.4 if precipitation > 2.0 else 1.0) * 0.2 +
        (traffic_flow / 1000.0) * 0.3 +
        (1.5 if is_roadwork == 1 else 1.0) * 0.1
    )
    risk_score += np.random.normal(0, 0.1)
    target = 1 if risk_score > 1.15 else 0

    all_rows.append({
        'region': reg_name,
        'date': record_date,
        'location_name': loc['name'],
        'lat': loc['lat'],
        'lon': loc['lon'],
        'hour': hour,
        'day_of_week': day_of_week,
        'temperature': round(temperature, 1),
        'precipitation': precipitation,
        'traffic_flow': traffic_flow,
        'is_roadwork': is_roadwork,
        'target_incident': target
    })

df = pd.DataFrame(all_rows)

# 2. Interactive UI Widgets
region_widget = widgets.Dropdown(options=list(regions_data.keys()), value='Riverside', description='Target Suburb:', style={'description_width': '140px'})
model_widget = widgets.Dropdown(options=[('XGBoost', 'XGB'), ('Random Forest', 'RF'), ('Logistic Regression', 'LR')], value='XGB', description='AI Algorithm:', style={'description_width': '140px'})
period_widget = widgets.Dropdown(options=[('All Periods (2025-2026)', 'ALL'), ('Year 2025 Only', '2025'), ('Year 2026 Only', '2026'), ('Winter Season', 'WINTER'), ('Summer Season', 'SUMMER')], value='ALL', description='Historical Period:', style={'description_width': '140px'})

# Dynamic location dropdown based on region
location_widget = widgets.Dropdown(options=[], description='Intersection:', style={'description_width': '140px'})

def update_locations(change):
    selected_reg = region_widget.value
    locs = regions_data[selected_reg]
    location_widget.options = [(l['name'], i) for i, l in enumerate(locs)]

region_widget.observe(update_locations, names='value')
update_locations(None)

hour_widget = widgets.IntSlider(value=8, min=0, max=23, step=1, description='Hour of Day:', style={'description_width': '140px'})
day_widget = widgets.Dropdown(options=[('Monday', 0), ('Tuesday', 1), ('Wednesday', 2), ('Thursday', 3), ('Friday', 4), ('Saturday', 5), ('Sunday', 6)], value=0, description='Day of Week:', style={'description_width': '140px'})
temp_widget = widgets.FloatSlider(value=18.0, min=-15.0, max=35.0, step=0.5, description='Temp (°C):', style={'description_width': '140px'})
precip_widget = widgets.Dropdown(options=[('None', 0.0), ('Light Rain', 0.5), ('Heavy Rain', 2.5), ('Storm / Snow', 10.0)], value=0.0, description='Precipitation:', style={'description_width': '140px'})
flow_widget = widgets.IntSlider(value=800, min=100, max=1500, step=50, description='Traffic (veh/h):', style={'description_width': '140px'})
roadwork_widget = widgets.Checkbox(value=False, description='Roadwork Active')

out = widgets.Output()

def run_simulation(b=None):
    with out:
        clear_output(wait=True)

        # Filter by region and period
        reg = region_widget.value
        region_df = df[df['region'] == reg]

        p_val = period_widget.value
        if p_val == '2025':
            filtered_df = region_df[region_df['date'].apply(lambda d: d.year == 2025)]
        elif p_val == '2026':
            filtered_df = region_df[region_df['date'].apply(lambda d: d.year == 2026)]
        elif p_val == 'WINTER':
            filtered_df = region_df[region_df['date'].apply(lambda d: d.month in [12, 1, 2])]
        elif p_val == 'SUMMER':
            filtered_df = region_df[region_df['date'].apply(lambda d: d.month in [6, 7, 8])]
        else:
            filtered_df = region_df.copy()

        if len(filtered_df) < 30:
            filtered_df = region_df.copy()

        features = ['hour', 'day_of_week', 'temperature', 'precipitation', 'traffic_flow', 'is_roadwork']
        X = filtered_df[features]
        y = filtered_df['target_incident']

        # Select AI Model algorithm
        algo = model_widget.value
        if algo == 'XGB':
            model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
        elif algo == 'RF':
            model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
        else:
            model = LogisticRegression(random_state=42)

        model.fit(X, y)

        locs_list = regions_data[reg]
        selected_loc = locs_list[location_widget.value]

        input_data = pd.DataFrame([{
            'hour': hour_widget.value,
            'day_of_week': day_widget.value,
            'temperature': temp_widget.value,
            'precipitation': precip_widget.value,
            'traffic_flow': flow_widget.value,
            'is_roadwork': int(roadwork_widget.value)
        }])

        probability = model.predict_proba(input_data)[0][1] * 100

        print(f"🏙️ Suburb: {reg} | 🤖 AI Model: {model_widget.label} | 📅 Period: {period_widget.label}")
        print(f"📍 Location: {selected_loc['name']}")
        print(f"🚨 Predicted Hotspot Probability: {probability:.1f}%")
        if probability > 60:
            print("⚠️ WARNING: High risk! Patrol unit dispatch advised.")
        elif probability > 30:
            print("⚡ Moderate risk level.")
        else:
            print("✅ Stable traffic conditions.")

run_btn = widgets.Button(description='Run AI Analysis', button_style='success', icon='play')
run_btn.on_click(run_simulation)

display(widgets.VBox([
    widgets.HTML("<h3>Multi-Model Traffic Hotspot Predictor (Text Mode)</h3>"),
    region_widget,
    model_widget,
    period_widget,
    location_widget,
    widgets.HBox([hour_widget, day_widget]),
    widgets.HBox([temp_widget, precip_widget]),
    widgets.HBox([flow_widget, roadwork_widget]),
    run_btn,
    widgets.HTML("<hr>")
], layout=widgets.Layout(width='100%')), out)

run_simulation()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.9 MB/s eta 0:00:00


Output()